# Muse EEG Heads — Kaggle outline

Frozen encoders (**CBraMod** default publish path; ± **LaBraM**; **REVE** optional/experimental) + trainable **Head A** / **Head B**.

**Channels:** AF7, AF8, TP9, TP10.

**License:** open mix only — **NO LUNA**, **NO L-FAME (BY-NC)**, **NO SEED-VIG**. See `docs/LICENSE_NOTES.md`.


## 1. Setup

Helpers live in the private Kaggle Dataset **`windwerfer/muse-eeg-heads-src`** (not pasted into cells).

Add it under **Add data → Your datasets** (or keep the attached data source). Prefer `uv` for installs when available.


In [ ]:
# Setup — load helpers from Kaggle Dataset windwerfer/muse-eeg-heads-src
import os, sys, random, shutil, subprocess
from pathlib import Path

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
INPUT_SRC = Path("/kaggle/input/muse-eeg-heads-src")
SRC = WORKING / "src"
SRC.mkdir(parents=True, exist_ok=True)

if INPUT_SRC.exists():
    for f in INPUT_SRC.glob("*.py"):
        shutil.copy(f, SRC / f.name)
    print("loaded modules from", INPUT_SRC, "→", SRC)
else:
    local = Path(".").resolve()
    for cand in (local / "src", local.parent / "src"):
        if cand.exists():
            for f in cand.glob("*.py"):
                shutil.copy(f, SRC / f.name)
            print("loaded modules from", cand)
            break
    else:
        raise FileNotFoundError(
            "Missing muse-eeg-heads-src dataset. "
            "Add data source windwerfer/muse-eeg-heads-src."
        )

sys.path.insert(0, str(WORKING))
ROOT = WORKING

need = []
for mod, pipname in [("numpy", "numpy")]:
    try:
        __import__(mod)
    except ImportError:
        need.append(pipname)
if need:
    if subprocess.call(["bash", "-lc", "command -v uv >/dev/null"]) == 0:
        subprocess.check_call(["uv", "pip", "install", "--system", "-q", *need])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print("ROOT", ROOT)
print("DEVICE", DEVICE)
from src.channel_map import MUSE_CHANNELS
from src.metrics import HEAD_A_LABELS, HEAD_B_LABELS
print("MUSE", MUSE_CHANNELS)
print("Head A", HEAD_A_LABELS)
print("Head B", HEAD_B_LABELS)


## 2. HF secret

Add Kaggle secret `HF_TOKEN` under **Add-ons → Secrets**, then **tick the checkbox** so this notebook can read it.

Required for Hugging Face downloads (CBraMod is public; REVE gated later). Never paste tokens into cells.


In [ ]:
# HF secret — diagnose attachment (never print the token)
import os
from kaggle_secrets import UserSecretsClient

def get_hf_token():
    try:
        secrets = UserSecretsClient()
    except Exception as e:
        print("UserSecretsClient init failed:", type(e).__name__, e)
        return os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

    # List secret NAMES only (helps confirm attachment)
    for meth in ("get_secrets", "list_secrets", "secrets"):
        try:
            fn = getattr(secrets, meth, None)
            if callable(fn):
                names = fn()
                print(f"secrets via {meth}:", names)
                break
            elif fn is not None:
                print(f"secrets attr {meth}:", fn)
                break
        except Exception as e:
            print(f"{meth} failed:", type(e).__name__, e)

    candidates = (
        "HF_TOKEN",
        "HUGGINGFACE_HUB_TOKEN",
        "HUGGING_FACE_TOKEN",
        "hf_token",
        "huggingface",
    )
    for key in candidates:
        try:
            tok = secrets.get_secret(key)
            if tok:
                print(f"found secret under name {key!r} (len={len(tok)})")
                return tok
            else:
                print(f"secret {key!r} empty")
        except Exception as e:
            print(f"get_secret({key!r}):", type(e).__name__, e)
    return os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

token = get_hf_token()
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = token
    print("HF token set (length=%d)" % len(token))
else:
    print("WARNING: no HF token — public downloads only; gated REVE base will fail (expected for default CBraMod path)")


## 3. Downloads / cache

Prefer the private dataset **`windwerfer/muse-eeg-heads-cache`** (CBraMod weights + Sleep-EDF pilot).

Attach under **Add data → Your datasets**. This cell copies from `/kaggle/input/muse-eeg-heads-cache` when present, and only network-fetches missing pieces.

**DENY:** LUNA, L-FAME, SEED-VIG.


In [ ]:
# Cache-first downloads — ALLOW only; paced if network needed
import os, time, json, shutil, subprocess, sys
from pathlib import Path

DATA = Path("/kaggle/working/data")
MODELS = Path("/kaggle/working/models")
DATA.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

CACHE = Path("/kaggle/input/muse-eeg-heads-cache")
DENY = {"LUNA", "L-FAME", "SEED-VIG", "luna", "l-fame", "seed-vig"}

DO_CBRAMOD = True
DO_SLEEP_EDF_PILOT = True
DO_EEGMEDITATION = False  # card-only in cache; full EEG later after license check
DO_OPENNEURO = False
FORCE_NETWORK = False  # True = ignore cache and re-fetch

PAUSE_SEC = 3.0

def pause(msg="…"):
    print(f"[pace] {msg} (sleep {PAUSE_SEC}s)")
    time.sleep(PAUSE_SEC)

def disk_free_gb(path="/kaggle/working"):
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / (1024**3)

def ensure_hf_hub():
    try:
        import huggingface_hub  # noqa: F401
        return
    except ImportError:
        pass
    pkgs = ["huggingface_hub", "hf_transfer"]
    if subprocess.call(["bash", "-lc", "command -v uv >/dev/null"]) == 0:
        subprocess.check_call(["uv", "pip", "install", "--system", "-q", *pkgs])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

def copytree_if(src: Path, dst: Path):
    if not src.exists():
        return False
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    return True

print(f"free disk ~{disk_free_gb():.1f} GB")
print("cache present:", CACHE.exists(), CACHE)
assert disk_free_gb() > 2, "Need more free disk"

# --- seed from cache dataset ---
cbramod_path = None
if CACHE.exists() and not FORCE_NETWORK:
    print("seeding from muse-eeg-heads-cache …")
    if DO_CBRAMOD:
        src = CACHE / "models" / "CBraMod"
        dst = MODELS / "CBraMod"
        if copytree_if(src, dst):
            w = dst / "pretrained_weights.pth"
            if w.exists() and w.stat().st_size > 1_000_000:
                cbramod_path = str(w)
                print(f"cached CBraMod -> {w} ({w.stat().st_size/1e6:.1f} MB)")
    if DO_SLEEP_EDF_PILOT:
        src = CACHE / "data" / "sleep-edfx-pilot"
        dst = DATA / "sleep-edfx-pilot"
        if copytree_if(src, dst):
            edfs = list(dst.rglob("*.edf"))
            print(f"cached Sleep-EDF pilot: {len(edfs)} edf files")
    # optional card peek
    em_src = CACHE / "data" / "EEGMeditation"
    if em_src.exists():
        copytree_if(em_src, DATA / "EEGMeditation")
        print("cached EEGMeditation card peek")

# --- 1) CBraMod weights (network fallback) ---
if DO_CBRAMOD and (cbramod_path is None or FORCE_NETWORK):
    pause("before CBraMod HF download")
    ensure_hf_hub()
    from huggingface_hub import hf_hub_download, login
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if tok:
        try:
            login(token=tok, add_to_git_credential=False)
            print("HF login: ok")
        except Exception as e:
            print("HF login soft-fail:", type(e).__name__, e)
    else:
        print("No HF_TOKEN in env — trying public download")
    out_dir = MODELS / "CBraMod"
    out_dir.mkdir(parents=True, exist_ok=True)
    repo = "weighting666/CBraMod"
    assert not any(d.lower() in repo.lower() for d in DENY)
    cbramod_path = hf_hub_download(repo_id=repo, filename="pretrained_weights.pth", local_dir=str(out_dir))
    size_mb = Path(cbramod_path).stat().st_size / (1024**2)
    print(f"CBraMod weights -> {cbramod_path} ({size_mb:.1f} MB)")
    (out_dir / "ATTRIBUTION.txt").write_text(
        "CBraMod pretrained weights from Hugging Face weighting666/CBraMod\n"
        "Upstream license: Apache-2.0 — attribute; do not re-license incorrectly.\n"
    )

# --- 2) Sleep-EDF pilot (network fallback) ---
sleep_dir = DATA / "sleep-edfx-pilot"
cass = sleep_dir / "sleep-cassette"
need_sleep = DO_SLEEP_EDF_PILOT and (
    FORCE_NETWORK or not cass.exists() or sum(1 for _ in cass.glob("*.edf")) < 4
)
if need_sleep:
    pause("before Sleep-EDF pilot")
    sleep_dir.mkdir(parents=True, exist_ok=True)
    cass.mkdir(parents=True, exist_ok=True)
    import urllib.request
    base = "https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette"
    files = [
        "SC4001E0-PSG.edf",
        "SC4001EC-Hypnogram.edf",
        "SC4002E0-PSG.edf",
        "SC4002EC-Hypnogram.edf",
    ]
    for f in files:
        url = f"{base}/{f}"
        dest = cass / f
        if dest.exists() and dest.stat().st_size > 1_000_000:
            print("exists", dest.name)
            continue
        print("GET", url)
        urllib.request.urlretrieve(url, dest)
        sz = dest.stat().st_size
        if sz < 1_000_000:
            dest.unlink(missing_ok=True)
            raise RuntimeError(f"Sleep-EDF download too small ({sz} B) for {f}")
        print(f"ok {f} ({sz/1e6:.1f} MB)")
        pause(f"after {f}")
    (sleep_dir / "ATTRIBUTION.txt").write_text(
        "Sleep-EDF Database Expanded (PhysioNet)\n"
        "https://physionet.org/content/sleep-edfx/\n"
        "License: Open Data Commons Attribution (ODC-By) — attribute; follow PhysioNet terms.\n"
        "Pilot subset only for pipeline bring-up.\n"
    )
    print(f"Sleep-EDF pilot: {len(list(sleep_dir.rglob('*.edf')))} edf files")
elif DO_SLEEP_EDF_PILOT:
    print("Sleep-EDF pilot already present — skip network")
else:
    print("skip Sleep-EDF pilot")

# --- 3) EEGMeditation (optional; off by default until license clear) ---
if DO_EEGMEDITATION:
    pause("before EEGMeditation")
    ensure_hf_hub()
    from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files, dataset_info
    ds_id = "alexeykashevnik/EEGMeditation"
    assert not any(d.lower() in ds_id.lower() for d in DENY)
    try:
        info = dataset_info(ds_id, token=os.environ.get("HF_TOKEN"))
        print("EEGMeditation license field:", getattr(info, "license", None))
    except Exception as e:
        print("dataset_info soft-fail:", type(e).__name__, e)
    em_dir = DATA / "EEGMeditation"
    files = list_repo_files(ds_id, repo_type="dataset", token=os.environ.get("HF_TOKEN"))
    print(f"EEGMeditation files ({len(files)})")
    if len(files) > 200:
        for fn in files:
            if fn.lower().endswith(("readme.md", "readme.txt", ".parquet", ".csv", ".json")):
                hf_hub_download(ds_id, fn, repo_type="dataset", local_dir=str(em_dir),
                                token=os.environ.get("HF_TOKEN"))
                break
    else:
        snapshot_download(ds_id, repo_type="dataset", local_dir=str(em_dir),
                          token=os.environ.get("HF_TOKEN"))
    (em_dir / "ATTRIBUTION.txt").write_text(
        f"Dataset: {ds_id}\nVerify open license on the HF dataset card before shipping training mix.\n"
    )
    print("EEGMeditation ->", em_dir)
else:
    print("skip EEGMeditation full download (license field was empty — verify before enabling)")

if DO_OPENNEURO:
    print("OpenNeuro not enabled in this cell")
else:
    print("skip OpenNeuro full dumps")

manifest = {
    "cbramod_weights": cbramod_path,
    "data_root": str(DATA),
    "models_root": str(MODELS),
    "cache_used": CACHE.exists() and not FORCE_NETWORK,
    "deny": sorted(DENY),
    "free_disk_gb_after": round(disk_free_gb(), 2),
}
(Path("/kaggle/working") / "download_manifest.json").write_text(json.dumps(manifest, indent=2))
print("manifest:", json.dumps(manifest, indent=2))
print("downloads cell done")


## 4. Preprocess (Sleep-EDF pilot → Muse proxy)

**Real Sleep-EDF preprocess/windowing lives in kernel** [`muse-eeg-heads-sleep-edf-windows`](https://www.kaggle.com/code/windwerfer/muse-eeg-heads-sleep-edf-windows) (`notebooks/02_sleep_edf_windows.ipynb`).

This outline keeps **cache/download + stubs only**. Proxy map (for reference): AF7=AF8=Fpz-Cz, TP9=TP10=Pz-Oz → 256 Hz, bandpass 1–45 Hz; slice around first N1.


In [ ]:
# Tiny smoke — verify pilot EDFs present (full pipeline → notebook 02)
from pathlib import Path
from src.sleep_edf import find_pilot_pairs, PROXY_NOTE

TARGET_SR = 256  # Hz — used by notebook 02 / export metadata
DATA = Path("/kaggle/working/data")
PILOT = DATA / "sleep-edfx-pilot"

pairs = find_pilot_pairs(PILOT)
assert pairs, f"No PSG/Hypnogram pairs under {PILOT} — is muse-eeg-heads-cache attached?"
print(f"found {len(pairs)} night(s) (outline smoke only)")
for psg, hyp in pairs[:2]:
    print(" ", psg.name, "+", hyp.name)
print(PROXY_NOTE)
print("→ run muse-eeg-heads-sleep-edf-windows for load/resample/window artifacts")
print("preprocess smoke OK")


## 5. Windowing + Head A labels from hypnogram

**Full windowing + Head A labels** (2.0 s / 0.5 s hop, W→`drowsy`, N1→`hypnagogic`) is in **`muse-eeg-heads-sleep-edf-windows`**.

Outline below keeps **synthetic stub windows** so Head A/B train stubs still run without duplicating the Sleep-EDF pipeline here.


In [ ]:
# Stub windows for outline train cells (real windows → notebook 02)
from collections import Counter
import numpy as np
from src.metrics import HEAD_A_LABELS, HEAD_B_LABELS
from src.windowing import stack_windows

WINDOW_SEC = 2.0
HOP_SEC = 0.5
# tiny synthetic (4, T) @ 256 Hz — not Sleep-EDF; placeholders for stub heads
rng = np.random.default_rng(42)
T = int(TARGET_SR * 8)  # ~8 s
x_stub = rng.normal(0, 1, size=(4, T)).astype(np.float32)
wins_a = stack_windows(x_stub, TARGET_SR, window_sec=WINDOW_SEC, hop_sec=HOP_SEC)
# alternate drowsy / hypnagogic stubs
labs_a = ["drowsy" if i % 2 == 0 else "hypnagogic" for i in range(len(wins_a))]
print("Head A stub windows", wins_a.shape, "label counts", Counter(labs_a))
assert wins_a.ndim == 3 and wins_a.shape[1] == 4
assert set(labs_a) <= set(HEAD_A_LABELS)

labs_b = ["clean"] * len(labs_a)
print("Head B placeholders", Counter(labs_b))

def make_windows(arr, sfreq=TARGET_SR, window_sec=WINDOW_SEC, hop_sec=HOP_SEC):
    return stack_windows(arr, sfreq, window_sec=window_sec, hop_sec=hop_sec)

print("windowing stub OK — see notebook 02 for real Sleep-EDF labels")


## 6. Head A train (Attention + Vigilance)

Labels: `concentration`, `mind_wandering`, `drowsy`, `hypnagogic`.

Encoder: **frozen CBraMod** (default). Optional LaBraM. REVE only if user supplied gated base.


In [ ]:
# Head A train stub — outline uses stub windows (real Sleep-EDF → notebook 02)
import numpy as np
from src.metrics import HEAD_A_LABELS

HEAD_A = list(HEAD_A_LABELS)
ENCODER_NAME = "CBraMod"  # default publish path; alt: "LaBraM"; REVE = experimental
label_to_id = {n: i for i, n in enumerate(HEAD_A)}

class FrozenEncoderStub:
    """Replace with real CBraMod/LaBraM forward; requires_grad_(False)."""
    def __init__(self, dim=256):
        self.dim = dim
    def encode(self, batch):
        # batch: (B, C, T) -> (B, dim)  — placeholder zeros
        b = batch.shape[0]
        return np.zeros((b, self.dim), dtype=np.float32)

class HeadAStub:
    def __init__(self, in_dim, n_classes=4):
        self.in_dim = in_dim
        self.n_classes = n_classes
    def fit(self, Z, y):
        print(f"[stub] Head A fit on {len(y)} windows, classes={HEAD_A}, encoder={ENCODER_NAME} frozen")
        print("  y counts:", {HEAD_A[i]: int((y==i).sum()) for i in range(len(HEAD_A))})
        return self
    def predict(self, Z):
        # majority-class baseline from training labels if available
        return np.zeros(len(Z), dtype=int)

enc = FrozenEncoderStub()
# subsample for stub speed
n_take = min(512, len(labs_a))
Xb = wins_a[:n_take]
ya = np.asarray([label_to_id[l] for l in labs_a[:n_take]], dtype=int)
Z = enc.encode(Xb)
head_a = HeadAStub(enc.dim).fit(Z, ya)
pred_a = head_a.predict(Z)
print("Head A preds sample", pred_a[:8], "true", ya[:8])


## 7. Head B train (artifacts)

Labels: `blink`, `double_blink`, `jaw`, `double_jaw`, `clean`.


In [ ]:
# Head B train stub — Sleep-EDF has no blink/jaw events; keep clean placeholders
import numpy as np
from src.metrics import HEAD_B_LABELS

HEAD_B = list(HEAD_B_LABELS)
label_to_id_b = {n: i for i, n in enumerate(HEAD_B)}

class HeadBStub:
    def __init__(self, in_dim, n_classes=5):
        self.in_dim = in_dim
        self.n_classes = n_classes
    def fit(self, Z, y):
        print(f"[stub] Head B fit on {len(y)} windows, classes={HEAD_B}, encoder={ENCODER_NAME} frozen")
        print("  (need Muse calibration / marker dataset for real blink/jaw)")
        return self
    def predict(self, Z):
        return np.full(len(Z), label_to_id_b["clean"], dtype=int)

n_take = min(512, len(labs_b))
yb = np.asarray([label_to_id_b[l] for l in labs_b[:n_take]], dtype=int)
head_b = HeadBStub(enc.dim).fit(Z[:n_take], yb)
pred_b = head_b.predict(Z[:n_take])
print("Head B preds sample", pred_b[:8])


## 8. Metrics

Macro-F1 + per-class report (`src.metrics`).


In [ ]:
# Metrics on stub predictions (real labels from Sleep-EDF)
from src.metrics import macro_f1, per_class_report

f1_a = macro_f1(ya, pred_a, HEAD_A)
f1_b = macro_f1(yb, pred_b, HEAD_B)
print("Head A macro-F1 (stub encoder/head)", f1_a)
print("Head B macro-F1 (stub)", f1_b)
print("Head A report", per_class_report(ya, pred_a, HEAD_A))
print("Head B report", per_class_report(yb, pred_b, HEAD_B))


## 9. Export (head-only)

Export **heads only** + label maps + attribution. Do not bundle gated REVE base or DENY-dataset-trained weights.


In [ ]:
# Export head-only stubs
import json
from datetime import datetime, timezone

OUT = Path("/kaggle/working/exports")
OUT.mkdir(parents=True, exist_ok=True)

card = {
    "format": "muse-eeg-heads/head-only",
    "encoder": ENCODER_NAME,
    "encoder_frozen": True,
    "channels": list(MUSE_CHANNELS),
    "head_a_labels": HEAD_A,
    "head_b_labels": HEAD_B,
    "window_sec": WINDOW_SEC,
    "hop_sec": HOP_SEC,
    "sample_rate": TARGET_SR,
    "datasets_allow_note": "Training mix must be ALLOW-only (no LUNA/L-FAME/SEED-VIG).",
    "publish_path": "CBraMod (± LaBraM); REVE experimental if user brings gated base",
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(OUT / "model_card_stub.json").write_text(json.dumps(card, indent=2))
(OUT / "ATTRIBUTION_STUB.md").write_text(
    "# Attribution\n\n"
    "- Encoder: %s (upstream license — do not re-publish gated full weights)\n"
    "- Datasets: list ALLOW corpora + SPDX/deed URLs used for this run\n"
    "- Heads: publish under MIT/BSD + attribution (preferred)\n" % ENCODER_NAME
)
# TODO: torch.save(head_a.state_dict(), OUT / "head_a.pt")
# TODO: torch.save(head_b.state_dict(), OUT / "head_b.pt")
print("Wrote", OUT / "model_card_stub.json")
print("Export complete (stubs) — replace with real head state_dicts before publish")
